# Mini-projet : le panier de courses vectorisé

| Module | Durée | Prérequis |
|---|---|---|
| M0 – Python et NumPy comme outils | 150 min | aucun |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Lina fait ses courses une fois par semaine pour sa colocation. Quatre magasins sont accessibles depuis chez elle. Elle a relevé les prix de sa liste habituelle (fichier `donnees/prix_magasins.csv`) et rassemblé quelques informations :

- **Hyper Nord** : ouvert de 8 h 30 à 21 h ; accessible en voiture, aller-retour estimé à **4,50 €** d'essence ; grand parking.
- **Super Centre** : ouvert de 7 h à 22 h ; deux tickets de tram, **1,20 €** au total ; programme de fidélité à points.
- **Épicerie du coin** : ouverte de 9 h à 20 h, fermée le lundi ; à pied, **0 €** ; le gérant est très sympathique.
- **Discount Sud** : ouvert de 9 h à 19 h 30 ; aller-retour en voiture, **3,00 €** ; ne vend **pas** de fromage râpé.

Sa liste de la semaine : 2 kg de pâtes, 1 kg de riz, 4 L de lait, 2 boîtes d'œufs, 2 kg de pommes, 1,5 kg de tomates, 2 sachets de fromage râpé, 1 paquet de café, 1 L d'huile d'olive, 1 paquet de pain de mie.

**Sa question : où faire ses courses ?**

**Activité** : Quelles informations ci-dessus sont utiles pour un calcul de coût ? Lesquelles ne le sont pas ? Lesquelles pourraient compter pour Lina, mais ne sont pas chiffrables ?

**Activité** : Reformulez la question de Lina en une ou deux questions précises.

**Activité** : Écrivez vos hypothèses. Par exemple : que faire du fromage râpé introuvable chez Discount Sud ? Lina peut-elle aller dans deux magasins ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_magasin = "Discount Sud"      # le magasin unique le moins cher, frais de trajet compris (modifiez)
pari_economie_deux_magasins = 2.0  # économie (€) en combinant deux magasins plutôt qu'un seul (modifiez)

## ③ Abstraire

Les prix forment un tableau $P$ : une ligne par produit (10), une colonne par magasin (4). La liste de courses est un vecteur $\mathbf{q}$ de 10 quantités. Les frais de trajet forment un vecteur $\mathbf{f}$ de 4 nombres.

**Activité** : Sur papier, dessinez le tableau $P$ et le vecteur $\mathbf{q}$. Comment obtient-on le coût du panier dans **un** magasin ? Et dans **tous** les magasins à la fois ?

**Activité** : Pour un panier réduit à 2 kg de pâtes et 4 L de lait, calculez le coût chez Hyper Nord et chez Super Centre. Vous comparerez avec le résultat de la machine.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Lire les données

In [ ]:
import numpy as np
import pandas as pd

tableau = pd.read_csv("donnees/prix_magasins.csv")
tableau

In [ ]:
magasins = list(tableau.columns[2:])
produits = list(tableau["produit"])
P = tableau[magasins].to_numpy()                          # forme (10, 4)
q = np.array([2, 1, 4, 2, 2, 1.5, 2, 1, 1, 1])           # dans l'ordre des produits du fichier
f = np.array([4.50, 1.20, 0.00, 3.00])                   # frais de trajet, même ordre que les magasins

print("P :", P.shape, " q :", q.shape, " f :", f.shape)
print("Cases vides dans P :", np.isnan(P).sum())

`nan` (*not a number*) marque une case vide : le fromage râpé chez Discount Sud. Toute opération avec `nan` donne `nan`. Il faut donc décider quoi faire : c'est une **hypothèse**, pas un détail technique.

### Scénario 1 : tout acheter dans un seul magasin

Hypothèse retenue ici : chez Discount Sud, Lina **renonce** au fromage râpé, puis l'achète à l'épicerie en rentrant (sans frais de trajet supplémentaire, l'épicerie est sur son chemin).

In [ ]:
P1 = P.copy()
i_fromage = produits.index("fromage râpé")
j_discount, j_epicerie = magasins.index("Discount Sud"), magasins.index("Épicerie du coin")
P1[i_fromage, j_discount] = P[i_fromage, j_epicerie]     # hypothèse : complément à l'épicerie

cout_panier = q @ P1                  # forme (4,) : un coût par magasin
cout_total = cout_panier + f

for m, c, t in zip(magasins, cout_panier, cout_total):
    print(f"{m:18s} panier {c:6.2f} €   + trajet = {t:6.2f} €")
print("\nMoins cher :", magasins[cout_total.argmin()])

**Corroborer.** Vérifiez votre calcul à la main du panier réduit (pâtes et lait) avec la machine.

In [ ]:
q_reduit = np.zeros(len(produits))
q_reduit[produits.index("pâtes")], q_reduit[produits.index("lait")] = 2, 4
print(dict(zip(magasins, (q_reduit @ P1).round(2).tolist())))

### Scénario 2 : combiner deux magasins

Lina peut aussi acheter chaque produit dans le moins cher de **deux** magasins, en payant les deux trajets. Il y a 6 paires possibles.

**Activité** : Comprenez la cellule suivante ligne par ligne. Quel est le rôle de `np.minimum` ? Pourquoi remplace-t-on `nan` par une valeur infinie ?

In [ ]:
from itertools import combinations

P2 = np.where(np.isnan(P), np.inf, P)     # un produit absent est « infiniment cher »

resultats = []
for j, k in combinations(range(len(magasins)), 2):
    meilleur_prix = np.minimum(P2[:, j], P2[:, k])   # le moins cher des deux, produit par produit
    total = q @ meilleur_prix + f[j] + f[k]
    resultats.append((total, magasins[j], magasins[k]))

for total, m1, m2 in sorted(resultats):
    print(f"{m1:18s} + {m2:18s} : {total:6.2f} €")

## ⑤ Interpréter

In [ ]:
import matplotlib.pyplot as plt

etiquettes = magasins + [f"{m1} + {m2}" for _, m1, m2 in resultats]
valeurs = list(cout_total) + [t for t, _, _ in resultats]
ordre = np.argsort(valeurs)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([etiquettes[i] for i in ordre][::-1], [valeurs[i] for i in ordre][::-1], color=LIOUAI["vert"])
ax.set_xlabel("coût total de la semaine, trajets compris (€)")
ax.set_title("Où Lina doit-elle faire ses courses ?")
plt.show()

meilleur_unique = cout_total.min()
meilleure_paire = min(resultats)[0]
print(f"Meilleur magasin unique : {magasins[cout_total.argmin()]} ({meilleur_unique:.2f} €) — votre pari : {pari_magasin}")
print(f"Économie en combinant deux magasins : {meilleur_unique - meilleure_paire:.2f} € — votre pari : {pari_economie_deux_magasins} €")

**Activité** : Rédigez la réponse à Lina en trois phrases au plus : où aller, combien elle dépense, combien elle économise par rapport aux autres options. Sur une année (52 semaines), que représente l'écart ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** Quelle hypothèse a le plus d'effet sur la conclusion : le coût du trajet, le traitement du fromage, ou le fait d'ignorer le temps passé ? Modifiez les frais de trajet de Discount Sud (par exemple 5 € au lieu de 3 €) : la conclusion change-t-elle ?
- **Généraliser.** Un cinquième magasin ouvre. Écrivez une fonction `meilleur_choix(P, q, f, noms)` qui renvoie le magasin unique et la paire les moins chers, quel que soit le nombre de magasins. Testez-la en ajoutant une colonne à `P`.

In [ ]:
def meilleur_choix(P, q, f, noms):
    # À compléter : renvoyer (meilleur magasin unique, meilleure paire) avec leurs coûts
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def meilleur_choix(P, q, f, noms):
    P = np.where(np.isnan(P), np.inf, P)
    unique = q @ P + f
    j = int(np.argmin(unique))
    paires = [(round(float(q @ np.minimum(P[:, a], P[:, b]) + f[a] + f[b]), 2), noms[a], noms[b])
              for a, b in combinations(range(P.shape[1]), 2)]
    return (noms[j], round(float(unique[j]), 2)), min(paires)

# Test avec un cinquième magasin fictif : « Marché du samedi », trajet à pied
marche = np.array([1.60, 2.50, 1.10, 1.80, 1.70, 1.90, 2.60, 3.60, 9.80, 1.90])
P5 = np.column_stack([P1, marche])
print(meilleur_choix(P5, q, np.append(f, 0.0), magasins + ["Marché du samedi"]))

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Filtrer l'information pertinente": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Transformer des constantes en paramètres": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M0-projet-panier", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")